# 🚀 RepoTopology: Fine-Tuning Gemma 4 31B with 4-Bit QLoRA
### Google - The Gemma 4 Developer Agent Competition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/repology/blob/main/colab_train_gemma4_31b.ipynb)

**Colab Pro Setup Instructions:**
1. Go to **Runtime** > **Change runtime type**
2. Select **A100 GPU** and **High-RAM**
3. Click **Runtime** > **Run all**

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

import os
import sys
import json
import shutil
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.models.gemma4 import modeling_gemma4
from trl import SFTConfig, SFTTrainer
from google.colab import files

# Fix: Align Gemma4ClippableLinear with nn.Linear so PEFT type-checks pass
modeling_gemma4.Gemma4ClippableLinear.__bases__ = (nn.Linear,)

# 2. Optional: Mount Google Drive for automatic backup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    has_drive = True
    print("Google Drive mounted successfully!")
except Exception as e:
    has_drive = False
    print("Drive mount skipped. Weights will be saved locally in session storage.")

MODEL_ID = "google/gemma-4-31b-it"
OUTPUT_DIR = "./adapters/main_lora"
DRIVE_BACKUP = "/content/drive/MyDrive/main_lora_31b.zip"

# 3. Load Tokenizer & Apply Chat Template
print(f"\n--- 1. Loading Tokenizer for {MODEL_ID} ---")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Locate & Format sft_trajectories.jsonl
print("\n--- 2. Loading sft_trajectories.jsonl ---")
TRAJ_FILE = None
for cand in ["/content/drive/MyDrive/repology_data/sft_trajectories.jsonl", "./data/sft_trajectories.jsonl", "sft_trajectories.jsonl", "/content/sft_trajectories.jsonl"]:
    if os.path.exists(cand):
        TRAJ_FILE = cand
        break

if not TRAJ_FILE:
    print("Trajectory file not found automatically. Please upload 'sft_trajectories.jsonl':")
    up = files.upload()
    for k in up:
        if k.endswith(".jsonl"):
            TRAJ_FILE = k
            break

formatted_data = []
with open(TRAJ_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            msgs = record.get("messages", [])
            if msgs:
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                formatted_data.append({"text": text})

print(f"Loaded {len(formatted_data)} trajectory decision samples.")
dataset = Dataset.from_list(formatted_data)

# 5. 4-bit Quantization Configuration (Compresses 31B to ~18 GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"\n--- 3. Loading 31B Base Model into A100 VRAM ---")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# Unwrap any ClippableLinear wrappers so PEFT accesses the inner Linear4bit layers
for name, module in model.named_modules():
    for child_name, child in module.named_children():
        if hasattr(child, 'linear') and 'ClippableLinear' in child.__class__.__name__:
            setattr(module, child_name, child.linear)

model = prepare_model_for_kbit_training(model)

# 6. LoRA Configuration (Rank 8, Alpha 16 matching competition specs)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# 7. SFT Training Arguments
training_args = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    gradient_checkpointing=True,
    bf16=True,
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# 8. Train
print("\n--- 4. Starting 31B Fine-Tuning (~8–12 mins on A100) ---")
trainer.train()

# 9. Save Final Adapter Weights
print(f"\n--- 5. Saving Adapter to {OUTPUT_DIR} ---")
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Update adapter_config.json to reflect the competition's quantized base model alias
config_path = f"{OUTPUT_DIR}/adapter_config.json"
with open(config_path, "r") as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = "google/gemma-4-31b-it-qat-w4a16-ct"
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

# 10. Archive for Download & Drive Backup
!zip -r /content/main_lora_31b.zip ./adapters/main_lora/

if has_drive:
    shutil.copy2("/content/main_lora_31b.zip", DRIVE_BACKUP)
    print(f"\n[SUCCESS] Backed up to Google Drive at: {DRIVE_BACKUP}")

print("\n[SUCCESS] Downloading main_lora_31b.zip to your local computer...")
files.download("/content/main_lora_31b.zip")
